### Setup

In [42]:
#!pip install pygleif
#!pip install lxml
#!pip install html5lib

In [43]:
#from pygleif import GleifClient
import pandas as pd
import numpy as np
import sys
pd.set_option('display.max_columns', None)
np.set_printoptions(threshold=sys.maxsize)


#fname = test_fname
hmda_lei_fname = r"C:\Users\brade\Documents\Grad School\Capstone\HMDA_Join_LEI.csv"
test_hmda_lei_fname = r"C:\Users\brade\Documents\Grad School\Capstone\Test_HMDA_Join_LEI.csv"
hmda_lei_fname = test_hmda_lei_fname


In [44]:
hmda_lei = pd.read_csv(hmda_lei_fname)
hmda_lei = hmda_lei.fillna(0)   #Treat all nan values as 0, so that np.where can compare floats for all numerical (non coded) fields, 
                                # and so that nan can be used to replace, then remove exempt loans.  Note, coded colums (used for amortization flag) can only be 1, 2 or 1111
hmda_lei = hmda_lei[hmda_lei["action_taken"] == 1] #Filter to only originated loans
hmda_lei = hmda_lei.replace("Exempt",np.nan) #Replace fields with the value "Exempt" with nan
hmda_lei = hmda_lei.replace(1111,np.nan) #Replace fields with the value 1111 (code for exempt for certain fields) with nan
hmda_lei = hmda_lei[["activity_year","lei","Legal Name","action_taken","rate_spread","intro_rate_period","total_points_and_fees","origination_charges","discount_points","lender_credits","negative_amortization","interest_only_payment","balloon_payment","other_nonamortizing_features","prepayment_penalty_term"]]
hmda_lei.dropna(inplace=True) #Remove all rows for loans that are exempt from reporting on fields used in this analysis
hmda_lei["Legal Name-LEI"] = hmda_lei["Legal Name"] + "-" + hmda_lei["lei"]

#Convert numeric columns from strings to floats
hmda_lei["rate_spread"] = hmda_lei["rate_spread"].astype(float)
hmda_lei["intro_rate_period"] = hmda_lei["intro_rate_period"].astype(float)
hmda_lei["total_points_and_fees"] = hmda_lei["total_points_and_fees"].astype(float)
hmda_lei["origination_charges"] = hmda_lei["origination_charges"].astype(float)
hmda_lei["discount_points"] = hmda_lei["discount_points"].astype(float)
hmda_lei["lender_credits"] = hmda_lei["lender_credits"].astype(float)
hmda_lei["prepayment_penalty_term"] = hmda_lei["prepayment_penalty_term"].astype(float)

print(len(hmda_lei))
hmda_lei.head()

67117


,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,total_points_and_fees,origination_charges,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI
9,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.000,0.0,1615.0,0.00,0.0,0.0,2.0,2.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G..."
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.0,0.00,0.0,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G..."
44,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.200,0.0,0.0,772.01,0.0,655.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325
46,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.564,0.0,0.0,0.00,0.0,1837.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325
48,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,-0.586,0.0,0.0,2484.48,0.0,650.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325


### Loan Level Predatory Behavior Flags

In [45]:
#Test_where_flag
#hmda_lei["test_where_flag"] = np.where((hmda_lei["derived_msa-md"] == 19340) & (hmda_lei["derived_loan_product_type"] == "FHA:First Lien"),True,False)

#hmda_lei.head()


In [46]:
#print(sorted(hmda_lei["rate_spread"].unique()))
#hmda_lei["rate_spread"].unique()

In [47]:
#Interest_Rate_Flag
hmda_lei["Rate_Spread_Flag"] = np.where((hmda_lei["rate_spread"] > 2),True,False)

print(hmda_lei[hmda_lei["rate_spread"] > 2]["Rate_Spread_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["rate_spread"] <= 2]["Rate_Spread_Flag"].unique() == False) #Should be False Only

#All tests above should return [ True]

print(min(hmda_lei[hmda_lei["Rate_Spread_Flag"]==False]["rate_spread"].unique()))
print(max(hmda_lei[hmda_lei["Rate_Spread_Flag"]==False]["rate_spread"].unique()))
print(min(hmda_lei[hmda_lei["Rate_Spread_Flag"]==True]["rate_spread"].unique()))
print(max(hmda_lei[hmda_lei["Rate_Spread_Flag"]==True]["rate_spread"].unique()))

#hmda_lei[hmda_lei["Rate_Spread_Flag"]==False].head(10)
#hmda_lei[hmda_lei["Rate_Spread_Flag"]==True].head(10)


[ True]
[ True]
-825.0
2.0
2.001
178.0


In [48]:
#hmda_lei["intro_rate_period"].unique()

In [49]:
#Adjustable_Rate_Flag
hmda_lei["Adjustable_Rate_Flag"] = np.where(hmda_lei["intro_rate_period"] > 0 ,True,False)

#hmda_lei.head()

#Non-Predatory Values
print(hmda_lei[hmda_lei["intro_rate_period"] == 0]["Adjustable_Rate_Flag"].unique() == False) #Should be False Only
 
#Predatory Values
print(hmda_lei[hmda_lei["intro_rate_period"] == 1]["Adjustable_Rate_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["intro_rate_period"] == 62]["Adjustable_Rate_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["intro_rate_period"] == 84]["Adjustable_Rate_Flag"].unique() == True) #Should be True Only



#All tests above should return [ True]

print(min(hmda_lei[hmda_lei["Adjustable_Rate_Flag"]==False]["intro_rate_period"].unique()))
print(max(hmda_lei[hmda_lei["Adjustable_Rate_Flag"]==False]["intro_rate_period"].unique()))
print(min(hmda_lei[hmda_lei["Adjustable_Rate_Flag"]==True]["intro_rate_period"].unique()))
print(max(hmda_lei[hmda_lei["Adjustable_Rate_Flag"]==True]["intro_rate_period"].unique()))

hmda_lei[hmda_lei["Adjustable_Rate_Flag"] == True].head()


[ True]
[ True]
[ True]
[ True]
0.0
0.0
1.0
360.0


,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,total_points_and_fees,origination_charges,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI,Rate_Spread_Flag,Adjustable_Rate_Flag
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.0,0.00,0.00,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",False,True
208,2025,549300HW662MN1WU8550,"UNITED WHOLESALE MORTGAGE, LLC",1,0.113,62.0,0.0,12549.98,10454.98,25.0,2.0,2.0,2.0,2.0,0.0,"UNITED WHOLESALE MORTGAGE, LLC-549300HW662MN1W...",False,True
345,2025,549300HW662MN1WU8550,"UNITED WHOLESALE MORTGAGE, LLC",1,0.953,84.0,0.0,6185.00,4095.00,190.1,2.0,2.0,2.0,2.0,0.0,"UNITED WHOLESALE MORTGAGE, LLC-549300HW662MN1W...",False,True
359,2025,549300HW662MN1WU8550,"UNITED WHOLESALE MORTGAGE, LLC",1,0.000,64.0,0.0,17682.34,7336.29,228.0,2.0,2.0,2.0,2.0,0.0,"UNITED WHOLESALE MORTGAGE, LLC-549300HW662MN1W...",False,True
388,2025,549300HW662MN1WU8550,"UNITED WHOLESALE MORTGAGE, LLC",1,-0.108,40.0,0.0,13448.36,11373.36,270.0,2.0,2.0,2.0,2.0,0.0,"UNITED WHOLESALE MORTGAGE, LLC-549300HW662MN1W...",False,True


In [50]:
#total_points_and_fees	origination_charges	discount_points	lender_credits

#print(hmda_lei["total_points_and_fees"].unique())
#print(hmda_lei["origination_charges"].unique())
#print(hmda_lei["discount_points"].unique())
#print(hmda_lei["lender_credits"].unique())

In [51]:
#Helper _Total_Points_And_Fees_Flag
hmda_lei["_Total_Points_And_Fees_Flag"] = np.where(hmda_lei["total_points_and_fees"].astype(float) >0 ,True,False)

#WRITE TESTS!!!
print(hmda_lei[hmda_lei["total_points_and_fees"] == 1615]["_Total_Points_And_Fees_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["total_points_and_fees"] == 0]["_Total_Points_And_Fees_Flag"].unique() == False) #Should be False

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["_Total_Points_And_Fees_Flag"]==False]["total_points_and_fees"].unique()))
print(max(hmda_lei[hmda_lei["_Total_Points_And_Fees_Flag"]==False]["total_points_and_fees"].unique()))
print(min(hmda_lei[hmda_lei["_Total_Points_And_Fees_Flag"]==True]["total_points_and_fees"].unique()))
print(max(hmda_lei[hmda_lei["_Total_Points_And_Fees_Flag"]==True]["total_points_and_fees"].unique()))



[ True]
[ True]
0.0
0.0
27.0
6315.0


In [52]:
#Helper _Origination_Charges_Flag
hmda_lei["_Origination_Charges_Flag"] = np.where(hmda_lei["origination_charges"].astype(float) >0 ,True,False)

#WRITE TESTS!!!
print(hmda_lei[hmda_lei["origination_charges"] == 772.01]["_Origination_Charges_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["origination_charges"] == 0]["_Origination_Charges_Flag"].unique() == False) #Should be False

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["_Origination_Charges_Flag"]==False]["origination_charges"].unique()))
print(max(hmda_lei[hmda_lei["_Origination_Charges_Flag"]==False]["origination_charges"].unique()))
print(min(hmda_lei[hmda_lei["_Origination_Charges_Flag"]==True]["origination_charges"].unique()))
print(max(hmda_lei[hmda_lei["_Origination_Charges_Flag"]==True]["origination_charges"].unique()))



[ True]
[ True]
0.0
0.0
0.01
37523.24


In [53]:
#Helper _Discount_Points_Flag

hmda_lei["_Discount_Points_Flag"] = np.where(hmda_lei["discount_points"].astype(float) >0 ,True,False)


#WRITE TESTS!!!
print(hmda_lei[hmda_lei["discount_points"] == 14132.61]["_Discount_Points_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["discount_points"] == 0]["_Discount_Points_Flag"].unique() == False) #Should be False

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["_Discount_Points_Flag"]==False]["discount_points"].unique()))
print(max(hmda_lei[hmda_lei["_Discount_Points_Flag"]==False]["discount_points"].unique()))
print(min(hmda_lei[hmda_lei["_Discount_Points_Flag"]==True]["discount_points"].unique()))
print(max(hmda_lei[hmda_lei["_Discount_Points_Flag"]==True]["discount_points"].unique()))

[ True]
[ True]
0.0
0.0
0.01
36028.24


In [54]:
#Helper _Lender_Credits_Flag

hmda_lei["_Lender_Credits_Flag"] = np.where(hmda_lei["lender_credits"].astype(float) >0 ,True,False)


#WRITE TESTS!!!
print(hmda_lei[hmda_lei["lender_credits"] == 655]["_Lender_Credits_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["lender_credits"] == 0]["_Lender_Credits_Flag"].unique() == False) #Should be False

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["_Lender_Credits_Flag"]==False]["lender_credits"].unique()))
print(max(hmda_lei[hmda_lei["_Lender_Credits_Flag"]==False]["lender_credits"].unique()))
print(min(hmda_lei[hmda_lei["_Lender_Credits_Flag"]==True]["lender_credits"].unique()))
print(max(hmda_lei[hmda_lei["_Lender_Credits_Flag"]==True]["lender_credits"].unique()))

[ True]
[ True]
0.0
0.0
0.01
14833.0


In [55]:
#Origination_Charge_Flag
hmda_lei["All_Origination_Charges_Flag"] = np.where((hmda_lei["_Total_Points_And_Fees_Flag"] == True) | (hmda_lei["_Origination_Charges_Flag"] == True) | 
                                                (hmda_lei["_Discount_Points_Flag"] == True) | (hmda_lei["_Lender_Credits_Flag"] == True),True,False)

hmda_lei.head()


,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,total_points_and_fees,origination_charges,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI,Rate_Spread_Flag,Adjustable_Rate_Flag,_Total_Points_And_Fees_Flag,_Origination_Charges_Flag,_Discount_Points_Flag,_Lender_Credits_Flag,All_Origination_Charges_Flag
9,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.000,0.0,1615.0,0.00,0.0,0.0,2.0,2.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",False,False,True,False,False,False,True
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.0,0.00,0.0,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",False,True,False,False,False,False,False
44,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.200,0.0,0.0,772.01,0.0,655.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,False,False,False,True,False,True,True
46,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.564,0.0,0.0,0.00,0.0,1837.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,False,False,False,False,False,True,True
48,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,-0.586,0.0,0.0,2484.48,0.0,650.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,False,False,False,True,False,True,True


In [56]:
#negative_amortization	interest_only_payment	balloon_payment	other_nonamortizing_features
#For all, code 1=exhibits behavior, code 2=does not exhibit behavior, code 1111=exempt

print(hmda_lei["negative_amortization"].unique())
print(hmda_lei["interest_only_payment"].unique())
print(hmda_lei["balloon_payment"].unique())
print(hmda_lei["other_nonamortizing_features"].unique())

[2. 1.]
[2. 1.]
[2. 1.]
[2. 1.]


In [57]:
#Non_Amortizing_Flag
hmda_lei["Non_Amortizing_Flag"] = np.where((hmda_lei["negative_amortization"] == 1) | (hmda_lei["interest_only_payment"] == 1) | (hmda_lei["balloon_payment"] == 1) | (hmda_lei["other_nonamortizing_features"] == 1),True,False)

hmda_lei.head()

#Non Predatory Behaviors - Should probably implement tests with multiple conditions given the OR nature of this flag, could check that sum of 4 cols is 8 if false, 3<x<8 if true

#Predatory Behaviors
print(hmda_lei[hmda_lei["negative_amortization"]==1]["Non_Amortizing_Flag"].unique()) #Should be True Only
print(hmda_lei[hmda_lei["interest_only_payment"]==1]["Non_Amortizing_Flag"].unique()) #Should be True Only
print(hmda_lei[hmda_lei["balloon_payment"]==1]["Non_Amortizing_Flag"].unique()) #Should be True Only
print(hmda_lei[hmda_lei["other_nonamortizing_features"]==1]["Non_Amortizing_Flag"].unique()) #Should be True Only

hmda_lei[["negative_amortization","interest_only_payment","balloon_payment","other_nonamortizing_features","Non_Amortizing_Flag"]].head(30)

hmda_lei[["negative_amortization","interest_only_payment","balloon_payment","other_nonamortizing_features","Non_Amortizing_Flag"]].head(30)


[ True]
[ True]
[ True]
[ True]


,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,Non_Amortizing_Flag
9,2.0,2.0,2.0,2.0,False
11,2.0,1.0,2.0,2.0,True
44,2.0,2.0,2.0,2.0,False
46,2.0,2.0,2.0,2.0,False
48,2.0,2.0,2.0,2.0,False
51,2.0,2.0,2.0,2.0,False
52,2.0,2.0,2.0,2.0,False
55,2.0,2.0,2.0,2.0,False
61,2.0,2.0,2.0,2.0,False
63,2.0,2.0,2.0,2.0,False


In [58]:
hmda_lei["prepayment_penalty_term"].unique()
#hmda_lei[pd.isna(hmda_lei["prepayment_penalty_term"])]["prepayment_penalty_term"].unique()

array([ 0., 24.,  2., 36., 12., 60.])

In [59]:
#Prepayment_Penalty_Flag
hmda_lei["Prepayment_Penalty_Flag"] = np.where((hmda_lei["prepayment_penalty_term"] > 0 ),True,False)

#hmda_lei.head()
#hmda_lei["Prepayment_Penalty_Flag"].unique()

#Non-Predatory Values
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 0]["Prepayment_Penalty_Flag"].unique() == False) #Should be False Only

#Predatory Values
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 24]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 36]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 2]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 12]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only
print(hmda_lei[hmda_lei["prepayment_penalty_term"] == 60]["Prepayment_Penalty_Flag"].unique() == True) #Should be True Only

#All tests above should return [ True]


print(min(hmda_lei[hmda_lei["Prepayment_Penalty_Flag"]==False]["prepayment_penalty_term"].unique()))
print(max(hmda_lei[hmda_lei["Prepayment_Penalty_Flag"]==False]["prepayment_penalty_term"].unique()))
print(min(hmda_lei[hmda_lei["Prepayment_Penalty_Flag"]==True]["prepayment_penalty_term"].unique()))
print(max(hmda_lei[hmda_lei["Prepayment_Penalty_Flag"]==True]["prepayment_penalty_term"].unique()))



[ True]
[ True]
[ True]
[ True]
[ True]
[ True]
0.0
0.0
2.0
60.0


In [60]:
#Loan-Level 
hmda_lei["Loan_Count_Predatory_Behaviors"] = hmda_lei[["Rate_Spread_Flag","Adjustable_Rate_Flag","All_Origination_Charges_Flag","Non_Amortizing_Flag","Prepayment_Penalty_Flag"]].sum(axis=1)

hmda_lei.head()

,activity_year,lei,Legal Name,action_taken,rate_spread,intro_rate_period,total_points_and_fees,origination_charges,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Legal Name-LEI,Rate_Spread_Flag,Adjustable_Rate_Flag,_Total_Points_And_Fees_Flag,_Origination_Charges_Flag,_Discount_Points_Flag,_Lender_Credits_Flag,All_Origination_Charges_Flag,Non_Amortizing_Flag,Prepayment_Penalty_Flag,Loan_Count_Predatory_Behaviors
9,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.000,0.0,1615.0,0.00,0.0,0.0,2.0,2.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",False,False,True,False,False,False,True,False,False,1
11,2025,7H6GLXDRUGQFU57RNE97,"JPMorgan Chase Bank, National Association",1,0.296,1.0,0.0,0.00,0.0,0.0,2.0,1.0,2.0,2.0,0.0,"JPMorgan Chase Bank, National Association-7H6G...",False,True,False,False,False,False,False,True,False,2
44,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.200,0.0,0.0,772.01,0.0,655.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,False,False,False,True,False,True,True,False,False,1
46,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,0.564,0.0,0.0,0.00,0.0,1837.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,False,False,False,False,False,True,True,False,False,1
48,2025,549300LYRWPSYPK6S325,FREEDOM MORTGAGE CORPORATION,1,-0.586,0.0,0.0,2484.48,0.0,650.0,2.0,2.0,2.0,2.0,0.0,FREEDOM MORTGAGE CORPORATION-549300LYRWPSYPK6S325,False,False,False,True,False,True,True,False,False,1


### Aggregation Functions

In [61]:
hmda_lei.groupby("Legal Name-LEI").mean("rate_spread")

,activity_year,action_taken,rate_spread,intro_rate_period,total_points_and_fees,origination_charges,discount_points,lender_credits,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,prepayment_penalty_term,Rate_Spread_Flag,Adjustable_Rate_Flag,_Total_Points_And_Fees_Flag,_Origination_Charges_Flag,_Discount_Points_Flag,_Lender_Credits_Flag,All_Origination_Charges_Flag,Non_Amortizing_Flag,Prepayment_Penalty_Flag,Loan_Count_Predatory_Behaviors
Legal Name-LEI,,,,,,,,,,,,,,,,,,,,,,,
21ST MORTGAGE CORPORATION-549300XQVJ1XBNFA5536,2025.0,1.0,4.633877,0.000000,427.238955,59.153769,0.000000,1.282836,2.0,2.000000,2.000000,2.0,0.0,0.820896,0.000000,0.108209,0.018657,0.000000,0.018657,0.126866,0.000000,0.0,0.947761
"A & N MORTGAGE SERVICES, INC.-549300ANF834QIIFUE57",2025.0,1.0,0.425500,6.000000,0.000000,1570.000000,0.000000,0.000000,2.0,2.000000,2.000000,2.0,0.0,0.000000,0.500000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.0,1.500000
A&D Mortgage LLC-2549005SIVTHG14U2905,2025.0,1.0,-0.003429,0.000000,0.000000,1871.413571,596.375000,777.698571,2.0,2.000000,2.000000,2.0,0.0,0.000000,0.000000,0.000000,0.500000,0.285714,0.285714,0.500000,0.000000,0.0,0.500000
ADVANTAGEFIRST LENDING INC.-254900M2C4Q6DATP2889,2025.0,1.0,0.417649,0.000000,0.000000,11293.788919,9369.552432,303.037568,2.0,2.000000,2.000000,2.0,0.0,0.000000,0.000000,0.000000,1.000000,1.000000,0.162162,1.000000,0.000000,0.0,1.000000
ALLIANT CREDIT UNION-549300T94GSH3C4U5M59,2025.0,1.0,0.142545,34.909091,0.000000,1447.068182,383.431818,710.559091,2.0,2.000000,2.000000,2.0,0.0,0.000000,0.363636,0.000000,1.000000,0.272727,0.727273,1.000000,0.000000,0.0,1.363636
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Westbury Bank-5493002XHCZ94X8YST31,2025.0,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.0,2.000000,1.000000,2.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.0,1.000000
"Wilmington Savings Fund Society, FSB-5493005DKMV1IHIM8E20",2025.0,1.0,0.873667,0.666667,0.000000,0.000000,0.000000,92.833333,2.0,1.333333,1.333333,2.0,0.0,0.000000,0.666667,0.000000,0.000000,0.000000,0.333333,0.333333,0.666667,0.0,1.666667
Wings Financial Credit Union-549300HVW3AI97UKTO72,2025.0,1.0,0.157286,65.285714,0.000000,1071.428571,0.000000,0.000000,2.0,2.000000,2.000000,2.0,0.0,0.142857,0.857143,0.000000,0.857143,0.000000,0.000000,0.857143,0.000000,0.0,1.857143
